# Stage 2 — Typological Conditioning (O2: FiLM, O3: Soft Prompts)

Investigates whether URIEL typological vectors can improve multilingual captioning beyond the O1-lang baseline.

## Setup

In [ ]:
import sys, os
sys.path.append("../src")

import torch
import torch.nn as nn
from transformers import AutoTokenizer
from models import ProjectionMLP, FiLMGenerator, TypologyPromptGenerator, apply_film, load_mt5_with_lora
from dataset import TRAINING_LANGUAGES, LANG_TO_ID, build_multilingual_loaders

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("google/mt5-base")

## URIEL Typological Vectors

Load the 103-dimensional URIEL syntactic feature vectors for all 7 training languages.

In [ ]:
# uriel_vectors.pt: tensor of shape (7, 103) — one row per training language
# Ordered by LANG_TO_ID: en=0, de=1, uk=2, ar=3, tr=4, bn=5, vi=6
uriel_vectors = torch.load("../uriel_vectors.pt", weights_only=True).to(device)
print(f"URIEL vectors: {uriel_vectors.shape}  (languages × features)")
print(f"Language order: {TRAINING_LANGUAGES}")

## Option A: FiLM Conditioning (O2)

Generates per-dimension γ and β from URIEL vector to modulate projected visual features.

> **Note:** Unconstrained γ growth (reaching 4–5× by epoch 10) degrades performance. `apply_film` clamps γ ∈ [0, 3.0] based on the observed feature distribution std.

In [ ]:
projection = ProjectionMLP().to(device)
mt5        = load_mt5_with_lora().to(device)
lang_emb   = nn.Embedding(len(TRAINING_LANGUAGES), 768).to(device)
film_gen   = FiLMGenerator().to(device)

# Load multilingual baseline checkpoint
ckpt = torch.load("../outputs/multilingual_best.pt", weights_only=True)
projection.load_state_dict(ckpt["projection"])
mt5.load_state_dict(ckpt["lora"])
lang_emb.load_state_dict(ckpt["lang_emb"])
print("O1-lang checkpoint loaded ✓")

In [ ]:
# Verify FiLM identity initialisation (γ=1, β=0 at epoch 0)
test_uriel = uriel_vectors[:2]
gamma, beta = film_gen(test_uriel)
print(f"γ mean: {gamma.mean().item():.4f}  (expected ~1.0)")
print(f"β mean: {beta.mean().item():.4f}   (expected ~0.0)")

In [ ]:
# Train O2-FiLM
# Full training via: python src/train.py --stage film --checkpoint outputs/multilingual_best.pt
# Below is a quick single-epoch sanity check

feature_files = {lang: f"../{lang}_features.pt" for lang in TRAINING_LANGUAGES}
train_loader, val_loader = build_multilingual_loaders(feature_files, tokenizer, batch_size=16)

models = {"projection": projection, "mt5": mt5, "lang_emb": lang_emb, "film_gen": film_gen}
optimizer = torch.optim.AdamW(
    [p for m in models.values() for p in m.parameters() if p.requires_grad],
    lr=1e-4, weight_decay=1e-2
)

# One batch sanity check
features, labels, lang_ids = next(iter(train_loader))
features, labels, lang_ids = features.to(device), labels.to(device), lang_ids.to(device)

uriel     = uriel_vectors[lang_ids]
gamma, beta = film_gen(uriel)
projected = apply_film(projection(features), gamma, beta)
enc_input = torch.cat([lang_emb(lang_ids).unsqueeze(1), projected], dim=1)

masked_labels = labels.clone()
masked_labels[masked_labels == tokenizer.pad_token_id] = -100
loss = mt5(inputs_embeds=enc_input, labels=masked_labels).loss
print(f"FiLM forward pass ✓  Loss: {loss.item():.4f}")

## Option B: Soft Prompt Conditioning (O3)

Maps URIEL vector to 8 learnable prompt tokens prepended to the encoder sequence. Unlike FiLM, this does not modify the visual feature distribution.

In [ ]:
projection2 = ProjectionMLP().to(device)
mt5_2       = load_mt5_with_lora().to(device)
lang_emb2   = nn.Embedding(len(TRAINING_LANGUAGES), 768).to(device)
prompt_gen  = TypologyPromptGenerator(n_prompts=8).to(device)

ckpt = torch.load("../outputs/multilingual_best.pt", weights_only=True)
projection2.load_state_dict(ckpt["projection"])
mt5_2.load_state_dict(ckpt["lora"])
lang_emb2.load_state_dict(ckpt["lang_emb"])
print("O1-lang checkpoint loaded ✓")

In [ ]:
# Sanity check: encoder input shape with prompts
test_features = torch.randn(4, 32, 768).to(device)
test_lang_ids = torch.zeros(4, dtype=torch.long).to(device)
test_uriel    = uriel_vectors[test_lang_ids]

prompts   = prompt_gen(test_uriel)                        # (4, 8, 768)
lang_tok  = lang_emb2(test_lang_ids).unsqueeze(1)         # (4, 1, 768)
projected = projection2(test_features)                    # (4, 32, 768)
enc_input = torch.cat([prompts, lang_tok, projected], dim=1)  # (4, 41, 768)

print(f"Encoder input shape: {enc_input.shape}  (B, 8 prompts + 1 lang + 32 visual, 768)")

## Full Training

Run O3 training via the training script:
```bash
python src/train.py --stage prompt --checkpoint outputs/multilingual_best.pt
```

Then evaluate:
```bash
python src/evaluate.py --stage prompt --checkpoint outputs/prompt_best.pt --langs en de ar tr bn vi
```

## FiLM Analysis

Inspect how γ and β correlate with URIEL typological similarity (Section 5.3 of the report).

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

# Load trained FiLM checkpoint
ckpt_film = torch.load("../outputs/film_best.pt", weights_only=True)
film_gen.load_state_dict(ckpt_film["film_gen"])
film_gen.eval()

with torch.no_grad():
    gammas = []
    for i in range(len(TRAINING_LANGUAGES)):
        g, _ = film_gen(uriel_vectors[i].unsqueeze(0))
        gammas.append(g.squeeze(0))
    gammas = torch.stack(gammas)  # (7, 768)

# Pairwise γ cosine similarity
gamma_sim = F.cosine_similarity(gammas.unsqueeze(0), gammas.unsqueeze(1), dim=-1)
uriel_sim = F.cosine_similarity(uriel_vectors.unsqueeze(0), uriel_vectors.unsqueeze(1), dim=-1)

# Correlation
mask = ~torch.eye(len(TRAINING_LANGUAGES), dtype=torch.bool)
corr = torch.corrcoef(torch.stack([gamma_sim[mask], uriel_sim[mask]]))[0, 1]
print(f"Pearson r (γ similarity vs URIEL similarity): {corr.item():.3f}")

# Plot
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(uriel_sim[mask].cpu(), gamma_sim[mask].cpu(), alpha=0.6)
ax.set_xlabel("URIEL cosine similarity")
ax.set_ylabel("γ cosine similarity")
ax.set_title(f"FiLM space vs typological space  (r={corr.item():.3f})")
plt.tight_layout()
plt.savefig("../outputs/film_uriel_correlation.png", dpi=150)
plt.show()